# RBI Policy Brief Generator with an LLM (and a Hallucination Check)
**What it does:** Reads an RBI policy document (PDF), uses an open-source **LLM** to write a short plain-English policy brief, compares a basic prompt with a well-designed prompt, and then **fact-checks the LLM's numbers** against the original text.

**Why it matters:** LLMs can sound confident and still make up numbers. In finance and economics, a wrong number is a serious problem. This project shows how to use an LLM *and* check it.

**Tech:** Python, Hugging Face Transformers, an open instruction-tuned LLM (Qwen2.5), PyTorch, pypdf, Google Colab. No API key needed.

**Tip:** Click *Runtime > Change runtime type > T4 GPU* before you start. It makes the LLM much faster. It also works on CPU, just slower.

## Step 1: Install libraries

In [1]:
!pip install -q pypdf transformers accelerate torch

## Step 2: Upload your PDF
Use an RBI Monetary Policy document (the Brickwork note, or the RBI resolution).

In [2]:
from google.colab import files
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print('Loaded:', pdf_name)

Saving Brickwork_Ratings_RBI_MPC_Meet_Outcome_October_2026_copy.pdf to Brickwork_Ratings_RBI_MPC_Meet_Outcome_October_2026_copy.pdf
Loaded: Brickwork_Ratings_RBI_MPC_Meet_Outcome_October_2026_copy.pdf


## Step 3: Read the PDF and split it into chunks
LLMs can only read a limited amount of text at once, so we split a long document into pieces of about 300 words.

In [3]:
from pypdf import PdfReader

reader = PdfReader(pdf_name)
text = ' '.join((p.extract_text() or '') for p in reader.pages)
text = ' '.join(text.split())

def make_chunks(text, size=300):
    words = text.split()
    return [' '.join(words[i:i + size]) for i in range(0, len(words), size)]

chunks = make_chunks(text)
print('Pages:', len(reader.pages), '| Words:', len(text.split()), '| Chunks:', len(chunks))

Pages: 12 | Words: 4555 | Chunks: 16


## Step 4: Load the LLM
`Qwen2.5-1.5B-Instruct` is a small open model that follows instructions. The first run downloads about 3 GB, so be patient. If it is too slow on CPU, change the name to `Qwen/Qwen2.5-0.5B-Instruct` (smaller and faster, but less accurate).

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using:', device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)
model.eval()

Using: cuda


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

## Step 5: A helper that asks the LLM something
A chat model takes a *system* message (its role and rules) and a *user* message (your request).

In [5]:
def ask_llm(system_msg, user_msg, max_new_tokens=250):
    messages = [
        {'role': 'system', 'content': system_msg},
        {'role': 'user', 'content': user_msg},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(ask_llm('You are a helpful assistant.', 'In one sentence, what is a repo rate?'))

A repo rate, also known as the reverse repo rate or overnight repurchase agreement rate, is the interest rate at which banks lend to each other in the interbank market.


## Step 6: Summarise the whole document (map-reduce)
A long document does not fit in one prompt, so we:
1. **Map:** summarise each chunk into 2 short bullets
2. **Reduce:** combine those bullets into one brief

To keep it fast we use only the first 6 chunks.

In [6]:
system_summary = ('You are an economics analyst. Use ONLY the text provided. '
                  'Do not add any number that is not in the text. Do not number your bullets.')

chunk_notes = []
for i, ch in enumerate(chunks[:6]):
    note = ask_llm(system_summary, 'Summarise this text in 2 short bullet points:\n\n' + ch, max_new_tokens=120)
    chunk_notes.append(note)
    print(f'Chunk {i + 1} done')

combined = '\n'.join(chunk_notes)
brief = ask_llm(system_summary,
                'Using these notes, write a policy brief of at most 120 words for an economics student. '
                'Cover the decision, the reasons, and the risks.\n\n' + combined,
                max_new_tokens=220)
print('\n=== POLICY BRIEF ===\n')
print(brief)

Chunk 1 done
Chunk 2 done
Chunk 3 done
Chunk 4 done
Chunk 5 done
Chunk 6 done

=== POLICY BRIEF ===

**Policy Brief: RBI Rate Hike**

RBI hiked the repo rate by 25 basis points to 5.50% on October 7, 2026, marking the first rate hike since February 2023. This action reflects a shift towards more aggressive monetary tightening, aligning with market expectations. Despite robust economic growth, the central bank aims to manage inflation effectively. The forecast projects near-term inflation averaging nearly 5.8% over the next three quarters, peaking at 6.0% in Q3 FY27. However, the base case scenario anticipates a mild economic downturn with core inflation stabilizing at 2.9%. 

Key considerations include maintaining control over liquidity and avoiding direct rate hikes, which could lead to capital outflows. Additionally, the current account deficit remains a significant concern, necessitating stringent capital controls. While credit conditions support strong growth absorption, they may d

## Step 7: Prompt engineering: basic prompt vs structured prompt
The same model gives different answers depending on how you ask. Compare them.

In [7]:
context = ' '.join(chunks[:3])
question = 'What did the central bank decide and why?'

basic = ask_llm('You are a helpful assistant.', question + '\n\n' + context, max_new_tokens=200)

structured_system = ('You are an economics analyst. Answer ONLY from the context. '
                     'If the answer is not in the context, say "Not stated in the text." '
                     'Format: Decision: ... | Reason: ... | Risk: ... Keep it under 80 words.')
structured = ask_llm(structured_system, 'Context:\n' + context + '\n\nQuestion: ' + question, max_new_tokens=200)

print('--- BASIC PROMPT ---\n', basic)
print('\n--- STRUCTURED PROMPT ---\n', structured)

--- BASIC PROMPT ---
 Based on the information provided, here's an overview of what happened during the October 2026 RBI monetary policy meeting:

**Key Points:**

1. **Repo Rate Hike:** The Reserve Bank of India (RBI) raised the repo rate by 25 basis points (bps) to 5.50%.

   - **Reason:** To combat inflationary pressures.
   
2. **Standing Deposit Facility (SDF) Rate:** The SDF rate increased to 5.25%.
   
3. **Marginal Standing Facility (MSF) Rate and Bank Rate:** Both rates were raised to 5.75%.
   
4. **Stance Change:** The RBI shifted from a neutral stance to a "calibrated tightening" stance.

5. **Inflation Expectations:** The RBI expected inflation to average around 5.8% over the next three quarters and reach a peak of 6.0% in Q3 FY27.

--- STRUCTURED PROMPT ---
 Decision: The Central Bank raised the repo rate by 25 basis points to 5.50%.
Reason: The RBI's Monetary Policy Committee (MPC) decided to raise the repo rate by 25 basis points to 5.50% on October 7, 2026, marking the

## Step 8: Hallucination check
For every number in the LLM's brief, check whether it appears in the original document. Numbers that do **not** appear are suspicious.

In [8]:
import re

def numbers_in(s):
    return set(re.findall(r'\d+(?:\.\d+)?', s))

source_numbers = numbers_in(text)

def check_numbers(llm_text, label):
    found = numbers_in(llm_text)
    found = {n for n in found if len(n) > 1}      # ignore single digits such as list numbers
    unsupported = sorted(n for n in found if n not in source_numbers)
    print(f'{label}: {len(found)} numbers checked, {len(unsupported)} not found in the source')
    if unsupported:
        print('  Suspicious numbers:', unsupported)
    return unsupported

check_numbers(brief, 'Policy brief')
check_numbers(basic, 'Basic prompt answer')
check_numbers(structured, 'Structured prompt answer')

Policy brief: 8 numbers checked, 0 not found in the source
Basic prompt answer: 8 numbers checked, 0 not found in the source
Structured prompt answer: 7 numbers checked, 0 not found in the source


[]

## Step 9: My analysis

**Number check:** The LLM did not introduce any numbers that were missing from the source. The policy brief, the basic prompt answer and the structured prompt answer all passed the check (0 unsupported numbers). However, the check cannot tell whether a real number was used in the right context.

**Prompt comparison:** I compared a basic prompt with a structured prompt (a role, rules, and a fixed output format) on the same question and the same text.

**Would I trust the brief without checking?** No. A small open model can still be vague or wrong, so I would read the brief against the source first.

## Limitations (important)
- A small open model is much weaker than the biggest LLMs and can still make mistakes
- The number check only finds numbers that are missing. A real number used in the wrong context would pass the check
- Only the first 6 chunks are summarised, to keep it fast
- Results can differ between runs and between PDFs